# 4. AIND Ephys Postprocessing

Build an `AINDEPhysPostprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run `AINDEPhysPostprocessingTask` for each coordinate. For every sorted recording the task removes duplicated units, builds a spikeinterface `SortingAnalyzer`, computes its extensions (waveforms, templates, locations, correlograms, template metrics, principal components, …) and quality metrics, and saves it to `postprocessed_<name>.zarr`.

The quality-metric bins assume minute-long recordings, so they are shortened for the 9 s toy recording.

## Imports

In [1]:
from pathlib import Path

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._04_postprocessing.blocks import QualityMetrics
from obi_one.scientific.tasks.aind_ephys._04_postprocessing.config import (
    AINDEPhysPostprocessingScanConfig,
)
from obi_one.scientific.tasks.aind_ephys.blocks import JobKwargs

## Build the scan config

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
preprocessing_output_path = obi_output / "02_aind_ephys_preprocessing" / "grid_scan" / "0"
spikesort_output_path = obi_output / "03_aind_ephys_spikesorting" / "grid_scan" / "0"
for path in (preprocessing_output_path, spikesort_output_path):
    assert path.exists(), f"{path} not found. Run notebooks 02 and 03 first."

scan_config = AINDEPhysPostprocessingScanConfig(
    initialize=AINDEPhysPostprocessingScanConfig.Initialize(
        preprocessing_output_path=preprocessing_output_path,
        spikesort_output_path=spikesort_output_path,
    ),
    job_kwargs=JobKwargs(n_jobs=1),
    quality_metrics=QualityMetrics(
        presence_ratio_bin_duration_s=1.0,
        firing_range_bin_size_s=1.0,
        amplitude_cv_min_num_bins=1,
        amplitude_cv_average_num_spikes_per_bin=5,
    ),
)

## Generate the grid scan and run the postprocessing task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "04_aind_ephys_postprocessing" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/core/core_tools.py:968: UserWarning: The extractor is not serializable to file. The provenance will not be saved.
  warnings.warn("The extractor is not serializable to file. The provenance will not be saved.")


/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/metrics/quality/misc_metrics.py:1006: UserWarning: Amplitude cutoff set to NaN for units [np.int64(0), np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]: too few spikes (< 500).
  warnings.warn(
/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packa

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/metrics/quality/misc_metrics.py:1303: UserWarning: Drift metrics could not be computed for units [np.int64(0), np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)] because they have less than 2 bins given the specified 'interval_s' and 'min_num_bins' or not enough valid intervals.
  warnings.warn(


[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/04_aind_ephys_postprocessing/grid_scan/0')]

## Load the analyzer and its quality metrics

`load_analyzer` skips the `principal_components` extension, whose fitted PCA models are pickled: numcodecs refuses to decode pickles since 0.15.2.

In [4]:
from obi_one.scientific.tasks.aind_ephys.utils import load_analyzer

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for analyzer_folder in sorted(coord_dir.glob("postprocessed_*.zarr")):
    analyzer = load_analyzer(analyzer_folder)
    print(analyzer)
    print("extensions:", sorted(analyzer.get_loaded_extension_names()))
    metrics = analyzer.get_extension("quality_metrics").get_data()
metrics[["num_spikes", "firing_rate", "snr", "presence_ratio", "rp_contamination", "amplitude_cutoff"]]

SortingAnalyzer: 31 channels - 7 units - 1 segments - zarr - sparse - has recording
Loaded 14 extensions: amplitude_scalings, correlograms, isi_histograms, noise_levels, quality_metrics, random_spikes, spike_amplitudes, spike_locations, template_metrics, template_similarity, templates, unit_locations, valid_unit_periods, waveforms
extensions: ['amplitude_scalings', 'correlograms', 'isi_histograms', 'noise_levels', 'quality_metrics', 'random_spikes', 'spike_amplitudes', 'spike_locations', 'template_metrics', 'template_similarity', 'templates', 'unit_locations', 'valid_unit_periods', 'waveforms']


,num_spikes,firing_rate,snr,presence_ratio,rp_contamination,amplitude_cutoff
0,388,43.110920,10.770405,1.0,0.320332,NaN
1,360,39.999822,23.009774,1.0,0.192054,NaN
3,41,4.555535,2.462117,1.0,1.000000,NaN
4,249,27.666544,27.591211,1.0,0.352419,NaN
5,154,17.111035,12.003358,1.0,0.000000,NaN
6,136,15.111044,8.107373,1.0,0.000000,NaN
7,123,13.666606,31.663763,1.0,0.000000,NaN
